# LangGraph meal-planning agent demo

Walks through the stateful LangGraph agent that computes macro deficits, scores recipes, and returns a ranked meal plan — all in-memory without MongoDB.

## StateGraph node diagram

```
┌───────────────┐
│  load_profile  │  ← validate user profile in state
└───────┬───────┘
        │
┌───────▼───────┐
│  load_fridge   │  ← validate fridge state in state
└───────┬───────┘
        │
┌───────▼────────────┐
│  compute_deficit    │  ← daily_targets − consumed → remaining budget
└───────┬────────────┘
        │
┌───────▼────────────┐
│  filter_recipes     │  ← dietary restrictions + fridge overlap
└───────┬────────────┘
        │
┌───────▼────────────┐
│  score_recipes      │  ← 0.7 × macro_fit + 0.3 × freshness_bonus
└───────┬────────────┘
        │
┌───────▼────────────┐
│  select_plan        │  ← top 3 (fallback if < 3 score > 0.2)
└───────┬────────────┘
        │
┌───────▼────────────┐
│  persist_log        │  ← write MealLog docs (skipped if error)
└───────┬────────────┘
        │
       END
```

All nodes are **deterministic Python functions** — no LLM calls.

## 1. Setup — in-memory objects

In [ ]:
from datetime import UTC, datetime

from db.models import (
    BoundingBox,
    DetectedItem,
    FridgeState,
    MacroTargets,
    MealItem,
    MealLog,
    MealType,
    UserProfile,
)

# Create a sample user profile
profile = UserProfile(
    user_id="demo_user",
    display_name="Demo User",
    daily_targets=MacroTargets(calories=2000, protein_g=60, carbs_g=250, fat_g=70),
    dietary_restrictions=["vegetarian"],
)

# Create a sample fridge state with items and freshness
now = datetime.now(tz=UTC)
fridge = FridgeState(
    user_id="demo_user",
    image_path="/demo/fridge.jpg",
    captured_at=now,
    detected_items=[
        DetectedItem(
            name="apple",
            bounding_box=BoundingBox(x_min=10, y_min=20, x_max=110, y_max=120),
            confidence=0.92,
            freshness_score=0.88,
        ),
        DetectedItem(
            name="banana",
            bounding_box=BoundingBox(x_min=120, y_min=30, x_max=220, y_max=130),
            confidence=0.87,
            freshness_score=0.65,
        ),
        DetectedItem(
            name="broccoli",
            bounding_box=BoundingBox(x_min=50, y_min=150, x_max=150, y_max=250),
            confidence=0.90,
            freshness_score=0.92,
        ),
        DetectedItem(
            name="tomato",
            bounding_box=BoundingBox(x_min=200, y_min=50, x_max=280, y_max=130),
            confidence=0.85,
            freshness_score=0.78,
        ),
    ],
)

print(f"User: {profile.display_name}")
print(f"Dietary restrictions: {profile.dietary_restrictions}")
print(f"Daily targets: {profile.daily_targets}")
print(f"Fridge items: {[item.name for item in fridge.detected_items]}")

## 2. Compute macro deficit

Simulates a user who already ate lunch (500 kcal, 20g protein, 50g carbs, 20g fat).

In [ ]:
from agent.tools import compute_macro_deficit

# Simulate today's consumed meals
lunch_log = MealLog(
    user_id="demo_user",
    meal_type=MealType.LUNCH,
    items=[
        MealItem(
            name="veggie sandwich",
            grams=250,
            calories=500,
            protein_g=20,
            carbs_g=50,
            fat_g=20,
        )
    ],
    logged_at=now,
)

deficit = compute_macro_deficit(profile.daily_targets, [lunch_log])

print("=== Macro Deficit (remaining budget) ===")
print(f"  Calories: {deficit.calories:.0f} kcal")
print(f"  Protein:  {deficit.protein_g:.1f} g")
print(f"  Carbs:    {deficit.carbs_g:.1f} g")
print(f"  Fat:      {deficit.fat_g:.1f} g")

## 3. Score individual recipes

Demonstrates the scoring formula:
```
macro_fit = 1 - mean_absolute_relative_error(recipe_macros, deficit)
freshness_bonus = mean freshness of matching fridge items
final_score = 0.7 × macro_fit + 0.3 × freshness_bonus
```

In [ ]:
from agent.tools import score_recipe

# Hand-pick two recipes to compare
recipe_good_match = {
    "id": "demo1",
    "name": "Apple Cinnamon Oatmeal",
    "macros": {"calories": 380, "protein_g": 14, "carbs_g": 62, "fat_g": 8},
    "uses_ingredients": ["apple"],
}

recipe_poor_match = {
    "id": "demo2",
    "name": "Heavy Cheese Pizza",
    "macros": {"calories": 1800, "protein_g": 60, "carbs_g": 180, "fat_g": 80},
    "uses_ingredients": [],
}

score_good = score_recipe(recipe_good_match, deficit, fridge)
score_poor = score_recipe(recipe_poor_match, deficit, fridge)

print(f"Apple Cinnamon Oatmeal  → score: {score_good:.4f}")
print(f"Heavy Cheese Pizza      → score: {score_poor:.4f}")
print(f"\nBetter match scores higher: {score_good > score_poor}")

## 4. Filter and score the full recipe corpus

In [ ]:
from agent.nodes import filter_recipes, score_recipes, select_plan
from agent.tools import load_recipes

# Load all 30 recipes from data/recipes.json
all_recipes = load_recipes()
print(f"Loaded {len(all_recipes)} recipes from data/recipes.json\n")

# Build an in-memory state
state = {
    "user_id": "demo_user",
    "user_profile": profile,
    "fridge_state": fridge,
    "meal_logs_today": [lunch_log],
    "macro_deficit": deficit,
    "candidate_recipes": [],
    "scored_recipes": [],
    "selected_plan": [],
    "projected_macros": None,
    "meal_type": "dinner",
    "error": None,
}

# Run filter_recipes node (uses load_recipes internally)
filter_result = filter_recipes(state)
state.update(filter_result)
print(f"Candidates after dietary filter: {len(state['candidate_recipes'])}")
for r in state["candidate_recipes"][:5]:
    print(f"  - {r['name']} (tags: {r['tags']})")

In [ ]:
# Run score_recipes node
score_result = score_recipes(state)
state.update(score_result)

print("=== All Scored Recipes (descending) ===")
for i, r in enumerate(state["scored_recipes"], 1):
    print(f"  {i:2d}. {r['name']:<40s}  score={r['_score']:.4f}")

## 5. Select top 3 meal plan

In [ ]:
# Run select_plan node
plan_result = select_plan(state)
state.update(plan_result)

print("=== Selected Meal Plan (Top 3) ===")
print()
for i, recipe in enumerate(state["selected_plan"], 1):
    m = recipe["macros"]
    print(f"{i}. {recipe['name']}")
    print(f"   Score: {recipe['_score']:.4f}")
    print(
        f"   Calories: {m['calories']} kcal | "
        f"Protein: {m['protein_g']}g | "
        f"Carbs: {m['carbs_g']}g | "
        f"Fat: {m['fat_g']}g"
    )
    print()

print("=== Projected Macros (sum of selected) ===")
pm = state["projected_macros"]
print(f"  Calories: {pm.calories:.0f} kcal")
print(f"  Protein:  {pm.protein_g:.1f} g")
print(f"  Carbs:    {pm.carbs_g:.1f} g")
print(f"  Fat:      {pm.fat_g:.1f} g")

print("\n=== Remaining Deficit After Plan ===")
print(f"  Calories: {max(0, deficit.calories - pm.calories):.0f} kcal")
print(f"  Protein:  {max(0, deficit.protein_g - pm.protein_g):.1f} g")
print(f"  Carbs:    {max(0, deficit.carbs_g - pm.carbs_g):.1f} g")
print(f"  Fat:      {max(0, deficit.fat_g - pm.fat_g):.1f} g")

## 6. Summary

This notebook demonstrated:

- **Macro deficit computation**: `daily_targets - consumed = remaining budget`
- **Recipe scoring**: `0.7 × macro_fit + 0.3 × freshness_bonus`
- **Dietary filtering**: vegetarian user sees only vegetarian/vegan recipes
- **Fridge overlap**: recipes using detected fridge items get priority
- **Plan selection**: top 3 recipes with projected macros

All logic runs as a **deterministic LangGraph state machine** — no LLM calls involved.